# 1 Atom Table

Flat view of every annotated atom across the corpus + the cross-article


In [1]:
import sys
import os
from pathlib import Path


def find_project_root(marker='src/data/atom_table.py'):
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find project root (no {marker} in any parent)")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 220)

from src.data.atom_table import (
    build_atom_table, summary_stats, tag_frequencies, show_atom,
    build_edge_table, add_edge_summary, article_connectivity,
    compute_tag_weights, show_tag_weights,
    compute_tag_similarities, build_semantic_edge_table, unique_tags_by_field,
    LIST_FIELDS,
)

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\gebruiker\Documents\Law & Tech


## 1. Load the atom table

In [2]:
df = build_atom_table()
print(f'{len(df)} atoms across {df["article_id"].nunique()} articles')
df.head()

49 atoms across 17 articles


,atom_id,article_id,lid,condition_type,text,text_nl,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual,source,annotator,annotated_at,notes
0,7:454(1).a,7:454,1,post_condition,The care provider sets up a file relating to the treatment of the ...,De hulpverlener richt een dossier in met betrekking tot de behande...,"[care provider, patient]",[treatment agreement],[sets up],[],[],[],[],"[file, treatment]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Patient tagged as essential non-acting actor per schema. Treatment...
1,7:454(1).b,7:454,1,post_condition,He keeps a record in the file of data concerning the health of the...,Hij houdt in het dossier aantekening van de gegevens omtrent de ge...,"[care provider, patient]",[],[keeps record],[],[],[],[],"[file, data, health]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Pronoun 'He' resolves to care provider. Treatment agreement droppe...
2,7:454(1).c,7:454,1,post_condition,and the interventions performed with respect to them,en de te diens aanzien uitgevoerde verrichtingen,"[care provider, patient]",[],[keeps record],[],[],[],[],[interventions],"Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Pronoun 'them' resolves to patient. Same act-verb as .b applied to...
3,7:454(1).d,7:454,1,post_condition,and includes therein other documents containing such data,"en neemt andere stukken, bevattende zodanige gegevens, daarin op",[care provider],[],[includes],[],[],[],[],"[documents, data, file]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,'Therein' refers to the file (residual). 'and' preserved per schem...
4,7:454(1).e,7:454,1,pre_condition,all this insofar as it is necessary for proper care to the patient,een en ander voor zover dit voor een goede hulpverlening aan de pa...,[patient],[],[],[],[],[],[],"[proper care, necessary]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Pre-condition (qualifier) limiting all duties in atoms .a-.d. 'Pro...


 ## 2. Per-article summary

In [3]:
summary_stats(df)

,article_id,n_atoms,n_pre,n_post,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual
0,7:454,8,2,6,7,2,6,0,3,1,1,8
1,7:455,3,2,1,2,1,1,0,1,1,2,3
2,synth:1,2,1,1,2,0,2,2,1,0,0,1
3,synth:10,3,2,1,2,0,3,0,0,0,1,3
4,synth:11,3,2,1,3,1,2,0,1,0,0,3
5,synth:12,2,1,1,1,0,2,1,1,0,0,2
6,synth:13,3,2,1,2,0,2,1,1,0,0,3
7,synth:14,2,1,1,1,0,2,1,1,0,0,2
8,synth:15,3,2,1,3,0,3,0,0,0,0,3
9,synth:2,2,1,1,1,0,1,0,1,0,0,2


## 3. Tag frequencies  how often each tag value appears; which concepts are the backbone of the corpus

In [4]:
for field in LIST_FIELDS:
    freqs = tag_frequencies(df, field)
    if len(freqs) == 0:
        continue
    print(f'\n--- {field} ---')
    print(freqs.to_string())


--- actors ---
actors
patient                                   17
care provider                              7
colonist                                   5
citizen                                    3
unemancipated minor                        2
someone other than patient                 1
certified emergency medical technician     1
attending physician                        1
lead epidemiologist                        1
orbital health authority                   1
legal guardian                             1
colonial medical board                     1
automated triage system                    1
treating specialist                        1
chief geneticist                           1
Extraterrestrial Health Directorate        1
clone vat operator                         1
orbital stem-cell research division        1
designated primary caregiver               1
sector magistrate                          1
backup medical proxy                       1
chief cybersecurity officer     

## 4. Build the edge table

One row per cross-article edge. Each edge is a single overlapping tag
Marks relation by two atoms basically sharing a term.

< 0.5 → low signal (edge from a common tag like patient)
0.5–1.5 → moderate (edge from a moderately rare tag)
 1.5 → strong (edge from a rare/distinctive tag)
 3.0 → very strong (two rare tags overlap)

In [5]:
edges = build_edge_table(df)
print(f'{len(edges)} cross-article edges')
edges

145 cross-article edges


,atom_a,atom_b,article_a,article_b,field,shared,strength,weighted_strength
0,7:454(1).a,7:455(1).a,7:454,7:455,actors,"[care provider, patient]",2,2.0
1,7:454(1).a,7:455(1).a,7:454,7:455,legal_relations,[treatment agreement],1,1.0
2,7:454(1).a,7:455(1).a,7:454,7:455,residual,[file],1,1.0
3,7:454(1).a,7:455(2).a,7:454,7:455,actors,[patient],1,1.0
4,7:454(1).a,synth:1(1).a,7:454,synth:1,actors,[patient],1,1.0
...,...,...,...,...,...,...,...,...
140,synth:11(1).a,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
141,synth:11(1).b,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
142,synth:11(1).c,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
143,synth:13(1).a,synth:15(1).a,synth:13,synth:15,actors,[colonist],1,1.0


## 5. Atom table with edge summary  <-- Reflects as cluster?

for every atom how many edges does it have,
which atoms is it connected to, and which schema fields produce those
edges.
Here we can see *hubs* (high `n_edges`)
and *isolates* (`n_edges == 0`).

In [6]:
df_e = add_edge_summary(df, edges)
df_e[['atom_id', 'condition_type', 'n_edges', 'connected_atoms', 'edge_fields']]

,atom_id,condition_type,n_edges,connected_atoms,edge_fields
0,7:454(1).a,post_condition,14,"[7:455(1).a, 7:455(2).a, synth:1(1).a, synth:1(1).b, synth:11(1).a...","[actors, legal_relations, residual]"
1,7:454(1).b,post_condition,14,"[7:455(1).a, 7:455(2).a, synth:1(1).a, synth:1(1).b, synth:11(1).a...","[actors, residual]"
2,7:454(1).c,post_condition,12,"[7:455(1).a, 7:455(2).a, synth:1(1).a, synth:1(1).b, synth:11(1).a...",[actors]
3,7:454(1).d,post_condition,3,"[7:455(1).a, 7:455(2).a]","[actors, residual]"
4,7:454(1).e,pre_condition,12,"[7:455(1).a, 7:455(2).a, synth:1(1).a, synth:1(1).b, synth:11(1).a...",[actors]
5,7:454(2).a,post_condition,15,"[7:455(1).a, 7:455(2).a, synth:1(1).a, synth:1(1).b, synth:11(1).a...","[actors, legal_relations, residual]"
6,7:454(3).a,post_condition,2,[7:455(1).a],"[actors, residual]"
7,7:454(3).b,pre_condition,0,[],[]
8,7:455(1).a,post_condition,24,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454...","[actors, legal_relations, residual]"
9,7:455(2).a,pre_condition,18,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454...","[actors, residual]"


Sort By connectiivity to spot huds --> Should reflect on graph


In [7]:
# Sort by connectivity to find the hubs
df_e.sort_values('n_edges', ascending=False)[['atom_id', 'n_edges', 'connected_atoms']]

,atom_id,n_edges,connected_atoms
8,7:455(1).a,24,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454..."
9,7:455(2).a,18,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454..."
43,synth:13(1).c,17,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455..."
11,synth:1(1).a,16,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455..."
12,synth:1(1).b,15,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455..."
21,synth:6(1).a,15,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455..."
29,synth:8(1).c,15,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455..."
22,synth:6(1).b,15,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455..."
5,7:454(2).a,15,"[7:455(1).a, 7:455(2).a, synth:1(1).a, synth:1(1).b, synth:11(1).a..."
28,synth:8(1).b,15,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455..."


In [8]:
# Isolated atoms — no cross-article edges yet
df_e.loc[df_e['n_edges'] == 0, ['atom_id', 'condition_type', 'text']]

,atom_id,condition_type,text
7,7:454(3).b,pre_condition,or as much longer as reasonably follows from the duty of a good ca...
10,7:455(2).b,pre_condition,nor insofar as any provision based on or by virtue of the law oppo...
13,synth:2(1).a,pre_condition,If a medical emergency occurs during an extravehicular surface tra...
14,synth:2(1).b,post_condition,any certified emergency medical technician may administer autonomo...
16,synth:3(1).b,post_condition,the orbital health authority must purge the specified data from al...
17,synth:4(1).a,pre_condition,If a pathogenic mutation is detected within a hydroponic agricultu...
19,synth:5(1).a,pre_condition,If an unemancipated minor undergoes artificial gravity acclimation...
20,synth:5(1).b,post_condition,the legal guardian may observe the physiological telemetry via a s...
23,synth:6(1).c,pre_condition,unless the pathogen is legally classified as a colony-threatening ...
25,synth:7(1).b,post_condition,the automated triage system must dispatch an emergency retrieval d...


## 6. Article-level connectivity

Aggregate edges to article we shud replace this with the graph after

In [9]:
article_connectivity(edges)

,article_a,article_b,n_edges,fields
0,7:454,7:455,22,"[actors, legal_relations, residual]"
1,7:454,synth:1,10,[actors]
2,7:454,synth:11,15,[actors]
3,7:454,synth:13,5,[actors]
4,7:454,synth:6,10,[actors]
5,7:454,synth:8,10,[actors]
6,7:455,synth:1,4,[actors]
7,7:455,synth:11,6,[actors]
8,7:455,synth:13,2,[actors]
9,7:455,synth:6,4,[actors]


## 7. Filter examples

In [10]:
# Which atoms share 'dossier' as a residual concept?
edges[edges['shared'].apply(lambda s: 'dossier' in s)]

,atom_a,atom_b,article_a,article_b,field,shared,strength,weighted_strength


In [11]:
# Edges by field — which channel produces the most edges? Right now its residual which is worrying
print(edges['field'].value_counts())

field
actors             132
residual             8
legal_relations      2
temporal             2
acts                 1
Name: count, dtype: int64


## 8. Tag weighting (discriminative weights)

Common tags  don't really distinguish
anything. IDF down-weights them and up-weights rare/discriminative tags.

Two modes:
- `'idf'` — rare tags get high weight (log inverse document frequency)
- `'tf'` — common tags get high weight (raw frequency)


In [12]:
# Compute IDF weights for every (field, tag) pair
weights_idf = compute_tag_weights(df, method='idf')
print(f'Computed {len(weights_idf)} tag weights')

# View all weights sorted high to low
show_tag_weights(weights_idf)


Computed 197 tag weights


,field,tag,weight
0,actors,someone other than patient,3.2189
1,actors,certified emergency medical technician,3.2189
2,actors,attending physician,3.2189
3,actors,lead epidemiologist,3.2189
4,actors,orbital health authority,3.2189
...,...,...,...
192,residual,data,2.3026
193,actors,colonist,2.1203
194,residual,file,1.9661
195,actors,care provider,1.8326


In [13]:
# The most common tags get the lowest weights:
show_tag_weights(weights_idf).tail(8)


,field,tag,weight
189,actors,citizen,2.5257
190,residual,request,2.5257
191,legal_relations,treatment agreement,2.5257
192,residual,data,2.3026
193,actors,colonist,2.1203
194,residual,file,1.9661
195,actors,care provider,1.8326
196,actors,patient,1.0217


### Weighted edges re-ranks the same 16 edges by discriminativeness

In [14]:
# Same engine, now passing weights
edges_weighted = build_edge_table(df, weights=weights_idf)
edges_weighted[['atom_a', 'atom_b', 'field', 'shared', 'strength', 'weighted_strength']]\
    .sort_values('weighted_strength', ascending=False)


,atom_a,atom_b,field,shared,strength,weighted_strength
57,7:454(2).a,7:455(1).a,residual,"[file, request]",2,4.4918
41,7:454(1).d,7:455(1).a,residual,"[data, file]",2,4.2687
15,7:454(1).b,7:455(1).a,residual,"[data, file]",2,4.2687
0,7:454(1).a,7:455(1).a,actors,"[care provider, patient]",2,2.8542
14,7:454(1).b,7:455(1).a,actors,"[care provider, patient]",2,2.8542
...,...,...,...,...,...,...
133,synth:8(1).c,synth:11(1).a,actors,[patient],1,1.0217
132,synth:8(1).b,synth:13(1).c,actors,[patient],1,1.0217
140,synth:11(1).a,synth:13(1).c,actors,[patient],1,1.0217
142,synth:11(1).c,synth:13(1).c,actors,[patient],1,1.0217


### What changed?

The same edges as before, but `weighted_strength` is now informative:


This ranking is what cluster-detection algorithms will use later: weighted edges produce more meaningful cluster boundaries than uniform edges.


### Compare uniform vs weighted ranking side by side

In [15]:
# Top 5 edges by each ranking
print('--- TOP 5 BY UNIFORM STRENGTH ---')
print(edges_weighted.sort_values('strength', ascending=False).head(5)[['atom_a', 'atom_b', 'shared', 'strength']].to_string(index=False))
print()
print('--- TOP 5 BY WEIGHTED STRENGTH (IDF) ---')
print(edges_weighted.sort_values('weighted_strength', ascending=False).head(5)[['atom_a', 'atom_b', 'shared', 'weighted_strength']].to_string(index=False))


--- TOP 5 BY UNIFORM STRENGTH ---
    atom_a     atom_b                   shared  strength
7:454(1).a 7:455(1).a [care provider, patient]         2
7:454(1).b 7:455(1).a [care provider, patient]         2
7:454(1).b 7:455(1).a             [data, file]         2
7:454(1).d 7:455(1).a             [data, file]         2
7:454(1).c 7:455(1).a [care provider, patient]         2

--- TOP 5 BY WEIGHTED STRENGTH (IDF) ---
    atom_a     atom_b                   shared  weighted_strength
7:454(2).a 7:455(1).a          [file, request]             4.4918
7:454(1).d 7:455(1).a             [data, file]             4.2687
7:454(1).b 7:455(1).a             [data, file]             4.2687
7:454(1).a 7:455(1).a [care provider, patient]             2.8542
7:454(1).b 7:455(1).a [care provider, patient]             2.8542


## 9. Semantic similarity (the new edge channel)

Exact set intersection misses synonyms and related concepts: `dossier` won't
match `gegevens`, `hulpverlener` won't match `zorgverlener`. The semantic
layer fixes that with a multilingual sentence-transformer. !*We changed the model to an english model.




In [16]:
 # Make sure sentence-transformers is installed:
#   pip install sentence-transformers
# Then load the model once (cached locally after first download)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print('Model loaded.')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded.


### Tag-level similarities

For every unique tag value in the corpus, compute cosine similarity with
every other tag in the same field. Keep pairs above the threshold.

Connects every unique word or phrase with others that carry similar meaning

In [17]:
sim_pairs = compute_tag_similarities(df, threshold=0.55, model=model)
print(f'{len(sim_pairs)} high-similarity tag pairs')
sim_pairs


32 high-similarity tag pairs


,field,tag_a,tag_b,similarity
0,residual,active medical emergency,medical emergency,0.7837
1,residual,civilian life-support telemetry,physiological telemetry,0.7407
2,temporal,after request,upon request,0.7242
3,residual,mandatory,necessary,0.7215
4,temporal,for twenty years,longer than twenty years,0.7192
5,actors,patient,someone other than patient,0.7153
6,actors,care provider,designated primary caregiver,0.7089
7,residual,physiological symptoms,symptoms,0.7069
8,actors,chief geneticist,chief xenobiologist,0.6940
9,acts,destroys,incinerates,0.6923


### Atom-level semantic edges !**Double Check**!

Promote tag-level similarities to atom-level edges: if atom A has tag X and atom B has tag Y in the same field, and X~Y is above threshold, then A and B have a semantic edge in that field.

In [18]:
sem_edges = build_semantic_edge_table(df, sim_pairs)
print(f'{len(sem_edges)} semantic edges across atoms')
sem_edges[['atom_a', 'atom_b', 'field', 'matched_pairs', 'n_matches', 'max_similarity']]


54 semantic edges across atoms


,atom_a,atom_b,field,matched_pairs,n_matches,max_similarity
0,7:454(1).a,7:455(2).a,actors,"[(patient, someone other than patient, 0.7153), (care provider, so...",2,0.7153
1,7:454(1).a,synth:11(1).b,actors,"[(care provider, designated primary caregiver, 0.7089)]",1,0.7089
2,7:454(1).b,7:455(2).a,actors,"[(patient, someone other than patient, 0.7153), (care provider, so...",2,0.7153
3,7:454(1).b,synth:1(1).b,acts,"[(keeps record, records, 0.5549)]",1,0.5549
4,7:454(1).b,synth:3(1).b,residual,"[(data, specified data, 0.6123)]",1,0.6123
5,7:454(1).b,synth:11(1).b,actors,"[(care provider, designated primary caregiver, 0.7089)]",1,0.7089
6,7:454(1).c,7:455(2).a,actors,"[(patient, someone other than patient, 0.7153), (care provider, so...",2,0.7153
7,7:454(1).c,synth:1(1).b,acts,"[(keeps record, records, 0.5549)]",1,0.5549
8,7:454(1).c,synth:11(1).b,actors,"[(care provider, designated primary caregiver, 0.7089)]",1,0.7089
9,7:454(1).d,7:455(2).a,actors,"[(care provider, someone other than patient, 0.5987)]",1,0.5987


### exact edges vs semantic edges

Both channels operate on the same atoms. Exact edges are deterministic and auditable; semantic edges catch synonyms exact-match misses. Use both.

In [19]:
# Cross-article exact edges from earlier
exact = edges.copy()
exact['edge_kind'] = 'exact'

# Cross-article semantic edges, harmonised to the same columns
sem = sem_edges.copy()
sem['edge_kind'] = 'semantic'
sem['shared'] = sem['matched_pairs']
sem['strength'] = sem['n_matches']
sem['weighted_strength'] = sem['max_similarity']

cols = ['edge_kind', 'atom_a', 'atom_b', 'field', 'shared', 'strength', 'weighted_strength']
combined = pd.concat([exact[cols], sem[cols]], ignore_index=True)
combined.sort_values(['atom_a', 'atom_b', 'edge_kind'])


,edge_kind,atom_a,atom_b,field,shared,strength,weighted_strength
0,exact,7:454(1).a,7:455(1).a,actors,"[care provider, patient]",2,2.0000
1,exact,7:454(1).a,7:455(1).a,legal_relations,[treatment agreement],1,1.0000
2,exact,7:454(1).a,7:455(1).a,residual,[file],1,1.0000
3,exact,7:454(1).a,7:455(2).a,actors,[patient],1,1.0000
145,semantic,7:454(1).a,7:455(2).a,actors,"[(patient, someone other than patient, 0.7153), (care provider, so...",2,0.7153
...,...,...,...,...,...,...,...
137,exact,synth:9(1).a,synth:13(1).a,actors,[colonist],1,1.0000
138,exact,synth:9(1).a,synth:15(1).a,actors,[colonist],1,1.0000
139,exact,synth:9(1).a,synth:15(1).b,actors,[colonist],1,1.0000
195,semantic,synth:9(1).b,synth:12(1).b,actors,"[(Extraterrestrial Health Directorate, lunar health minister, 0.55...",1,0.5506


### How much extra coverage does the semantic layer add?

In [20]:
print(f'Exact edges:     {len(exact)}')
print(f'Semantic edges:  {len(sem_edges)}')

# Which atom pairs only connect via semantic edges, not exact?
exact_pairs = {(r.atom_a, r.atom_b) for _, r in exact.iterrows()}
sem_pairs   = {(r.atom_a, r.atom_b) for _, r in sem.iterrows()}
sem_only    = sem_pairs - exact_pairs
exact_only  = exact_pairs - sem_pairs
both        = sem_pairs & exact_pairs
print(f'Atom pairs connected by exact only:    {len(exact_only)}')
print(f'Atom pairs connected by semantic only: {len(sem_only)}')
print(f'Atom pairs connected by both:          {len(both)}')

if sem_only:
    print()
    print('Atom pairs ONLY connected semantically:')
    for ap in sorted(sem_only):
        print(f'  {ap[0]} <-> {ap[1]}')


Exact edges:     145
Semantic edges:  54
Atom pairs connected by exact only:    110
Atom pairs connected by semantic only: 27
Atom pairs connected by both:          26

Atom pairs ONLY connected semantically:
  7:454(1).b <-> synth:3(1).b
  7:454(1).d <-> synth:11(1).b
  7:454(1).d <-> synth:3(1).b
  7:454(1).e <-> synth:12(1).b
  7:454(3).a <-> 7:455(2).a
  7:454(3).a <-> synth:11(1).b
  7:454(3).a <-> synth:15(1).b
  7:455(1).a <-> synth:10(1).b
  7:455(1).a <-> synth:3(1).b
  7:455(2).a <-> synth:3(1).b
  synth:10(1).c <-> synth:15(1).a
  synth:13(1).c <-> synth:14(1).b
  synth:2(1).a <-> synth:14(1).a
  synth:2(1).b <-> synth:12(1).a
  synth:3(1).a <-> synth:10(1).c
  synth:3(1).b <-> synth:9(1).b
  synth:4(1).a <-> synth:6(1).c
  synth:4(1).b <-> synth:12(1).b
  synth:4(1).b <-> synth:15(1).c
  synth:5(1).b <-> synth:11(1).b
  synth:5(1).b <-> synth:12(1).a
  synth:6(1).c <-> synth:9(1).c
  synth:7(1).a <-> synth:15(1).b
  synth:8(1).b <-> synth:15(1).b
  synth:8(1).c <-> synth:13

## Atom classifier

Given any atom-shaped input, rank the most similar atoms in the corpus and predict an article.

The score for each candidate atom is:
- **exact_score**: sum of IDF weights of tags shared exactly with the query
- **semantic_score**: sum of best semantic-similarity matches for tags not exactly matched
- **combined_score** = `alpha * exact_score + (1 - alpha) * semantic_score`

`alpha=0.5` weights both channels equally. Raise alpha to trust exact matching more; lower it to trust semantic more.

In [21]:
from src.data.atom_table import classify_atom

query = df[df['atom_id'] == '7:454(2).a'].iloc[0].to_dict()
result = classify_atom(query, df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=5)
print(f"Predicted article: {result['predicted_article']}")
result['top_matches']



Predicted article: 7:454


,atom_id,article_id,combined_score,exact_score,semantic_score
0,7:455(1).a,7:455,5.2980,9.8718,0.7242
1,7:454(1).a,7:454,3.6730,7.3461,0.0000
2,7:454(1).b,7:454,2.4102,4.8203,0.0000
3,7:455(2).a,7:455,2.0730,3.5474,0.5987
4,7:454(3).a,7:454,1.8993,3.7987,0.0000


Why the top match scored highest ; which exact and semantic matches contributed.

In [22]:
top_row = result['scores_by_atom'].iloc[0]
print(f"Top match: {top_row['atom_id']}")
print(f"  combined_score = {top_row['combined_score']}")
print(f"  exact_score    = {top_row['exact_score']}")
print(f"  semantic_score = {top_row['semantic_score']}")
print()
print('Exact matches (field, tag, weight):')
for m in top_row['exact_matches']:
    print(f'  {m}')
print()
print('Semantic matches (field, query_tag, candidate_tag, similarity):')
for m in top_row['semantic_matches']:
    print(f'  {m}')

Top match: 7:455(1).a
  combined_score = 5.298
  exact_score    = 9.8718
  semantic_score = 0.7242

Exact matches (field, tag, weight):
  ('actors', 'patient', 1.0217)
  ('actors', 'care provider', 1.8326)
  ('legal_relations', 'treatment agreement', 2.5257)
  ('residual', 'file', 1.9661)
  ('residual', 'request', 2.5257)

Semantic matches (field, query_tag, candidate_tag, similarity):
  ('temporal', 'upon request', 'after request', 0.7242)


### Classify a hand-written query

A hypothetical atom — not in the corpus — phrased like a paraphrase of 7:454(1).a. The classifier should rank that atom highly.

In [23]:
hand_query = {
    'atom_id': 'HAND_QUERY',
    'article_id': 'UNKNOWN',
    'actors': ['care provider', 'patient'],
    'legal_relations': ['treatment agreement'],
    'acts': ['sets up'],
    'residual': ['file', 'treatment'],
    'temporal': [], 'explicit_references': [], 'hierarchies': [], 'geographical_domain': [],
}
result = classify_atom(hand_query, df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=5)
print(f"Predicted article: {result['predicted_article']}")
result['top_matches']

Predicted article: 7:454


,atom_id,article_id,combined_score,exact_score,semantic_score
0,7:454(1).a,7:454,6.8919,13.7838,0.0
1,7:455(1).a,7:455,3.6730,7.3461,0.0
2,7:454(2).a,7:454,3.6730,7.3461,0.0
3,7:454(1).b,7:454,2.4102,4.8203,0.0
4,7:454(1).d,7:454,1.8993,3.7987,0.0


###  Leave-one-out evaluation

For each atom: remove it, classify it against the rest, check whether the correct article comes back at rank 1 / in top-k.

In [25]:
from src.data.atom_table import evaluate_classifier_loo

eval_df, summary = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=3)
print('Summary:')
for k, v in summary.items():
    print(f'  {k:25s} {v}')

Summary:
  n_atoms                   49
  top1_accuracy             0.2245
  top3_accuracy             0.3265
  mean_reciprocal_rank      0.2755


In [ ]:
eval_df[['atom_id', 'true_article', 'top1_match', 'top1_hit', 'rank']]


###  Per-article accuracy

In [26]:
per_article = eval_df.groupby('true_article').agg(
    n_atoms=('atom_id', 'size'),
    top1_accuracy=('top1_hit', 'mean'),
    topk_accuracy=('topk_hit', 'mean'),
    mrr=('reciprocal_rank', 'mean'),
).round(3)
per_article


,n_atoms,top1_accuracy,topk_accuracy,mrr
true_article,,,,
7:454,8,0.625,1.000,0.812
7:455,3,0.333,0.333,0.333
synth:1,2,0.000,0.000,0.000
synth:10,3,0.000,0.000,0.000
synth:11,3,0.000,0.000,0.000
synth:12,2,0.000,0.000,0.000
synth:13,3,0.000,0.000,0.000
synth:14,2,0.000,0.000,0.000
synth:15,3,0.333,0.333,0.333


### Tune `alpha`

Sweep from 0 (semantic only) to 1 (exact only).

In [27]:
sweep = []
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    _, s = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=sim_pairs, alpha=a, top_k=3)
    sweep.append({'alpha': a, **s})
pd.DataFrame(sweep)


,alpha,n_atoms,top1_accuracy,top3_accuracy,mean_reciprocal_rank
0,0.00,49,0.0816,0.1633,0.1156
1,0.25,49,0.2041,0.3469,0.2755
2,0.50,49,0.2245,0.3265,0.2755
3,0.75,49,0.2245,0.3265,0.2755
4,1.00,49,0.2245,0.3265,0.2755


###  Misclassified atoms

Atoms whose correct article didn't reach top-1 worth inspecting to see whether errors are intuitive (sibling articles look alike) or surprising.

In [28]:
misses = eval_df[~eval_df['top1_hit']]
misses[['atom_id', 'true_article', 'top1_match', 'rank']]


,atom_id,true_article,top1_match,rank
0,7:454(1).a,7:454,7:455,2.0
1,7:454(1).b,7:454,7:455,2.0
5,7:454(2).a,7:454,7:455,2.0
8,7:455(1).a,7:455,7:454,NaN
10,7:455(2).b,7:455,7:454,NaN
11,synth:1(1).a,synth:1,synth:12,NaN
12,synth:1(1).b,synth:1,7:454,NaN
13,synth:2(1).a,synth:2,synth:14,NaN
14,synth:2(1).b,synth:2,synth:12,NaN
15,synth:3(1).a,synth:3,synth:15,NaN


In [29]:
sweep5 = []
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    _, s = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=sim_pairs, alpha=a, top_k=5)
    sweep5.append({'alpha': a, **s})
pd.DataFrame(sweep5)

,alpha,n_atoms,top1_accuracy,top5_accuracy,mean_reciprocal_rank
0,0.00,49,0.0816,0.2449,0.1340
1,0.25,49,0.2041,0.3878,0.2837
2,0.50,49,0.2245,0.3878,0.2888
3,0.75,49,0.2245,0.3878,0.2878
4,1.00,49,0.2245,0.3673,0.2837
